## Import libraries

In [1]:

import pandas as pd
import numpy as np
import re
from datetime import datetime
from google_play_scraper import app, reviews, Sort
import sys
import os

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))
from scripts.text_cleaner import clean_text
from scripts.preprocessing import normalize_dates, remove_duplicates, validate_ratings

## Web Scraping

In [3]:
BOA_APP_ID = 'com.boa.boaMobileBanking'

# app metadata 
app_info = app(
    BOA_APP_ID,
    lang='en', 
    country='et'  
)

print("=" * 50)
print("BOA Bank App Info")
print("=" * 50)
print(f"App Title   : {app_info['title']}")
print(f"Current Score: {app_info['score']}")
print(f"Total Ratings: {app_info['ratings']:,}")
print(f"Total Reviews: {app_info['reviews']:,}")
print(f"Installs     : {app_info['installs']}")

BOA Bank App Info
App Title   : BoA Mobile
Current Score: 4.3925533
Total Ratings: 9,202
Total Reviews: 1,458
Installs     : 1,000,000+


## Collecting reviews

In [4]:
print(f"Scraping reviews for BOA...")

result, continuation_token = reviews(
    BOA_APP_ID,
    lang='en',
    country='et',
    sort=Sort.NEWEST,       
    count=500,              
    filter_score_with=None 
)

print(f"Collected {len(result)} raw reviews")

Scraping reviews for BOA...
Collected 500 raw reviews


## Inspecting data

In [5]:
print("Keys in a single review:")
print(list(result[0].keys()))

print("\nFirst raw review (sample):")
for key, value in result[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: ac90ba30-b2ea-4b38-b02d-a532b24afc4b
  userName: Muhammed ahmed
  userImage: https://play-lh.googleusercontent.com/a/ACg8ocKS2TmUoDH_M63owVePOjlo-xMHojmjri7k-0oyyLdivhkYXQ=mo
  content: muhammedahmed
  score: 5
  thumbsUpCount: 0
  reviewCreatedVersion: 26.05.11
  at: 2026-05-13 00:25:23
  replyContent: None
  repliedAt: None
  appVersion: 26.05.11


## Extracting needed fields

In [7]:
raw_data = []

for r in result:
    raw_data.append({
        'review_id': r.get('reviewId', ''),
        'review'   : r.get('content', ''),
        'rating'   : r.get('score', None),
        'date'     : r.get('at', None),
        'bank'     : 'Bank of Abyssinia',
        'source'   : 'Google Play'
    })

df = pd.DataFrame(raw_data)

print(f"Shape: {df.shape}")
df.head()

Shape: (500, 6)


,review_id,review,rating,date,bank,source
0,ac90ba30-b2ea-4b38-b02d-a532b24afc4b,muhammedahmed,5,2026-05-13 00:25:23,Bank of Abyssinia,Google Play
1,c3bb042c-844b-4580-98b9-df418622b2fb,it's very good app,5,2026-05-12 11:50:32,Bank of Abyssinia,Google Play
2,400ce769-3726-43b2-ac4d-755b3a15f026,this app is good but the speed of app is very ...,2,2026-05-11 18:18:54,Bank of Abyssinia,Google Play
3,4d6d2f22-5e71-47be-9cde-a1cf6c9fff93,good,5,2026-05-09 14:41:50,Bank of Abyssinia,Google Play
4,e77089b3-aecf-45e2-a64a-ce917fc4233a,boa the best,5,2026-05-08 13:47:07,Bank of Abyssinia,Google Play


## Exploring the Raw Data

In [8]:
print(f"Total reviews collected: {len(df)}")
print(f"\nColumn dtypes:")
print(df.dtypes)

Total reviews collected: 500

Column dtypes:
review_id            object
review               object
rating                int64
date         datetime64[ns]
bank                 object
source               object
dtype: object


## Rating distribution

In [9]:
print("Rating distribution:")
rating_counts = df['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts.items():
    bar = '█' * (count // 5)
    print(f"  {int(rating)} stars: {count:>4}  {bar}")

Rating distribution:
  5 stars:  280  ████████████████████████████████████████████████████████
  4 stars:   37  ███████
  3 stars:   18  ███
  2 stars:   16  ███
  1 stars:  149  █████████████████████████████


## Checking date column

In [10]:
print("Sample date values (raw):")
print(df['date'].head(10).to_string())

print(f"\nDate dtype: {df['date'].dtype}")

Sample date values (raw):
0   2026-05-13 00:25:23
1   2026-05-12 11:50:32
2   2026-05-11 18:18:54
3   2026-05-09 14:41:50
4   2026-05-08 13:47:07
5   2026-05-07 10:33:06
6   2026-05-05 11:03:08
7   2026-05-04 14:01:17
8   2026-05-03 13:40:13
9   2026-05-02 15:48:30

Date dtype: datetime64[ns]


## Missing Values

In [11]:
print("Missing Values")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

for col in df.columns:
    status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "Nothing missing :)"
    print(f"  {col:<15}: {status}")

Missing Values
  review_id      : Nothing missing :)
  review         : Nothing missing :)
  rating         : Nothing missing :)
  date           : Nothing missing :)
  bank           : Nothing missing :)
  source         : Nothing missing :)


## Duplicate Reviews

In [12]:
print("Duplicates")

exact_dupes = df.duplicated(subset=['review']).sum()
print(f"  Exact duplicate reviews : {exact_dupes}")

id_dupes = df.duplicated(subset=['review_id']).sum()
print(f"  Duplicate review IDs    : {id_dupes}")

empty_reviews = (df['review'].str.strip() == '').sum()
print(f"  Empty review texts      : {empty_reviews}")

Duplicates
  Exact duplicate reviews : 89
  Duplicate review IDs    : 0
  Empty review texts      : 0


## Remove Duplicates

In [13]:
df = remove_duplicates(df)

Removed 0 duplicate reviews


## Date Format

In [14]:
print("Date Format")
print(f"  Current dtype: {df['date'].dtype}")
print(f"  Sample values: {df['date'].iloc[0]}")
print(f"  Target format: YYYY-MM-DD (string or date object)")

Date Format
  Current dtype: datetime64[ns]
  Sample values: 2026-05-13 00:25:23
  Target format: YYYY-MM-DD (string or date object)


## Normalize dates

In [15]:
print("Before normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

df = normalize_dates(df)

print("\nAfter normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")

Before normalization:
0   2026-05-13 00:25:23
1   2026-05-12 11:50:32
2   2026-05-11 18:18:54
dtype: datetime64[ns]

After normalization:
0    2026-05-13
1    2026-05-12
2    2026-05-11
dtype: object

Date range: 2025-02-14 to 2026-05-13


## Clean Review Text

In [16]:
sample_raw = "  Great   app!\n\nVery useful.  "
print(f"Before: {repr(sample_raw)}")
print(f"After : {repr(clean_text(sample_raw))}")

# Apply to the full column
df['review'] = df['review'].apply(clean_text)

# Remove any reviews that became empty after cleaning
before = len(df)
df = df[df['review'].str.len() > 0]
removed = before - len(df)
print(f"\nRemoved {removed} reviews that were empty after cleaning")

Before: '  Great   app!\n\nVery useful.  '
After : 'Great app! Very useful.'

Removed 0 reviews that were empty after cleaning


## Rating Validation

In [17]:
df = validate_ratings(df)

print(f"Remaining: {len(df)} reviews")
print(f"Rating dtype: {df['rating'].dtype}")

Removed 0 invalid ratings
Remaining: 500 reviews
Rating dtype: int64


## Cleaned data

In [18]:
df_clean = df[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

Final dataset shape: (500, 5)


,review,rating,date,bank,source
0,muhammedahmed,5,2026-05-13,Bank of Abyssinia,Google Play
1,it's very good app,5,2026-05-12,Bank of Abyssinia,Google Play
2,this app is good but the speed of app is very ...,2,2026-05-11,Bank of Abyssinia,Google Play
3,good,5,2026-05-09,Bank of Abyssinia,Google Play
4,boa the best,5,2026-05-08,Bank of Abyssinia,Google Play
5,bank of absiniya is best bank in ethiopian,5,2026-05-07,Bank of Abyssinia,Google Play
6,አስተማማኝና ዘመኑን የዋጀ,4,2026-05-05,Bank of Abyssinia,Google Play
7,good,5,2026-05-04,Bank of Abyssinia,Google Play
8,extremely slow app and unreliable for most pay...,2,2026-05-03,Bank of Abyssinia,Google Play
9,Amazing app,5,2026-05-02,Bank of Abyssinia,Google Play


## Save cleaned data

In [19]:
output_path = '../data/processed/boa_reviews_clean.csv'
df_clean.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: ../data/processed/boa_reviews_clean.csv
